In [2]:
"""
This script is responsible for data acquisition by filtering and processing data 
from an SQL database using predefined queries. The retrieved data is then used 
for further calculations in subsequent processing steps.
"""

import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

# Third-party libraries
import requests  # For making HTTP requests
import pandas as pd  # For data manipulation
import sqlalchemy  # For database interaction
from sqlalchemy.engine import URL, create_engine
from shapely import wkb  # For geospatial data processing

# This is used to cache function return values (SQL query text → return value)
from joblib import Memory 

# Custom code and numerical/visualization tools
import numpy as np  # For numerical operations
import matplotlib.pyplot as plt  # For data visualization
import seaborn as sns  # For Seaborn-based plots
import plotly.express as px  # For interactive visualizations using Plotly
import geopandas as gpd  # For geospatial data analysis
from matplotlib.dates import HourLocator, DateFormatter
from utilities import run_sql  # Custom SQL execution utility

# Determine module path (handles Jupyter Notebook and Python script cases)
try:
    MODULE_PATH = Path(__file__).resolve().parent.parent  # Standard for scripts
    SCRIPT_PATH = Path(__file__).resolve()
except NameError:
    MODULE_PATH = Path.cwd().parent  # Jupyter Notebook case
    SCRIPT_PATH = MODULE_PATH / "data_acquisition.ipynb"  # Default fallback for Jupyter

if MODULE_PATH.as_posix() not in sys.path:
    sys.path.append(MODULE_PATH.as_posix())

# Setup caching for SQL queries (cache directory: `.cache/`)
CACHE_DIR = MODULE_PATH / ".cache"
CACHE_DIR.mkdir(exist_ok=True)  # Ensure cache directory exists
memory = Memory(CACHE_DIR, verbose=0)


@memory.cache
def cached_run_sql(query: str):
    """
    Executes an SQL query and caches the result.

    :param query: SQL query string
    :return: Query result (cached)
    """
    return run_sql(query)


def get_git_last_modified_info(file_path: Path) -> str:
    """
    Retrieves the last modification date and author of the given file using Git.

    :param file_path: Path to the script file
    :return: A formatted string containing the last modified timestamp and author
    """
    try:
        # Ensure Git is available
        subprocess.run(["git", "--version"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # Check if file exists in Git history (works even if modified)
        git_check = subprocess.run(
            ["git", "log", "--", str(file_path)],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )

        if git_check.returncode != 0 or not git_check.stdout.strip():
            return f"Warning: This file ({file_path}) is not yet committed to Git. Run 'git commit' to track changes."

        # Retrieve last commit info affecting this file
        git_log = subprocess.check_output(
            ["git", "log", "-1", "--format=%cd | %an", "--", str(file_path)],
            universal_newlines=True
        ).strip()

        # Convert to readable format
        timestamp, author = git_log.split(" | ")
        timestamp = datetime.strptime(timestamp, "%a %b %d %H:%M:%S %Y %z")

        return f"Last modified: {timestamp.strftime('%Y-%m-%d %H:%M:%S')} by {author}"

    except subprocess.CalledProcessError:
        return "Error: Unable to retrieve Git information. Ensure the file is committed."
    except FileNotFoundError:
        return "Error: Git is not installed or not available in PATH."
    except Exception as e:
        return f"Error: {e}"



# Execution (only when run as a script, not when imported)
if __name__ == "__main__":
    last_modified_info = get_git_last_modified_info(SCRIPT_PATH)
    print(last_modified_info)

